### Introduction and problem formulation

_What does the dataset (ds) represent?_
+ The ds represents of aerial photographs of buildings in bird's eye view.

_What does the ds contain?_
+ The ds contains RGB images in ``png`` format and corresponding grey-scale binary labels also in ``png``, representing rooftops of buildings in the images.

_What is the task at hand?_
+ Image segmentation: given the RGB image input, predict the binary mask → translates to a pixel-wise classification problem.

_What defines a good solution? Is there a specific metric (and/or related objectives) given?_
+ No specific metric given
+ A good solution encompasses a model that is 
    - better than random
    - identifies semantic regions, e.g connected objects
    - (optional) model accurately identifies edges of roofs

_Is there additional domain knowledge (of the physical system) we can use?_
- these additional constraints can typically help in data augmentation strategies and model selection.
- the problem is __invariant__ under physical changes such as 
    + daytime / nighttime
    + sun (and shadow) positioning
    + cloudy / non-cloudy days, etc.
- the problem is __equivariant__ under 
    + translations, rotations, reflections (flipping) → $E(2)$ group.

![Figure 1](figures/data_grid.png)

### Exploratory Data Analysis

#### 1. Data Quality assessment

_Are there obvious problems with the dataset?_

**High level assessment**:

+ Representativeness of the dataset
    - very low data regime (only 25 samples to train)
+ Data uniformity
    - Is the spatial resolution varying i.e. are the images taken from different same heights?
        - __No__ (based on visual inspection)
    - Are there distortions, i.e does the camera angle vary between images?
        - __No__ (based on visual inspection)
    - Do colors represent the same semantic information, i.e. are images taken at the same time of day and illumination settings?
        - __No__ → will be used in data augmentation strategy

**Low level assessment**:

+ Are there problems with the labels (i.e. from the conversion from png to binary)?
    - __Yes__, information png label masks very noisy at edges, need cleaning.
+ Are there corrupted values, Infs or NaNs?
    - __Yes__, cutout artifacts with apparent transparent fill → set to 0
+ Are there mismatched image-label pairs?
    - __Yes__, sample #278 label is the same as #270 → discard

#### 2. Data Cleaning

<u>Images</u>: Cleaning includes conversion of transparent pixels to background pixels (black, i.e. 0-valued). \
\
<u>Labels</u>: Label cleaning is done by mapping potential transparent pixels to background pixels, followed by a binarization transform to boolean masks using a threshold ``threshold_value`` = 0.9 (aggressive). Moreover, we perform opening and closing operations as morphological clean-up with the a kernel size ``morph_size`` = 3. \
\
<u>Masks</u>: We extract valid regions as masks (where alpha > 0.0). This information can be used during inference as these regions are always part of the background. They follow the same cleanup procedure as for the labels.

Let's see the effect of varying the ``threshold_value``:

<img src="figures/threshold_effect.png" width="600">

<img src="figures/morph_effect.png" width="600">

Lets visualize the effect of ``threshold_value``and ``morph_size`` on the rooftop size distributions:

<img src="figures/threshold_effect_hist.png" width="500">

<img src="figures/morph_effect_hist.png" width="500">

The final cleaning was performed with ``threshold = 0.9`` and ``morph_size = 3``.

#### 3. Basic correlation analysis

##### 3.1 Pixel-based feature analysis

Let's analyze the basic correlations between colors and labels:

<img src="figures/color_distributions.png" width="800">

The RGB values are not discmiminative enough for a simple classification. Based on the RGB values we can investigate the following additional features and their respective discriminative power:
+ luminance (discriminate bright from darker objects)
$$C_{\max} = \max(R,G,B), \quad
C_{\min} = \min(R,G,B), \quad
L = C_{\max} - C_{\min}$$
+ hsv color space (hue, saturation, value)
+ chromaticity
$$r = \frac{R}{R + G + B}, \quad
g = \frac{G}{R + G + B}, \quad
b = \frac{B}{R + G + B}$$
$$r + g + b = 1$$
+ excess green luminance (useful for identifying vegetation)
$$\mathrm{ExG} = 2G - R - B$$

From the pairplot below we see that, while there are some tendencies of disentanglement of the binary labels (e.g. excess green luminance vs hue), there is a considerable overlap remaining.

![Figure 7](figures/feature_pairplot.png)

##### 3.2 UMAP based clustering analysis
The above analysis indicates that individual features as well as linear combinations thereof (e.g. PCA) can not disentangle enough differently labelled data points, UMAP provides a non-linear approach to learn a (non-linear) embedding. So, let's analyze whether UMAP can find combinations of features that are predictive on a pixel-by-pixel basis. Below we show a (labe-balanced!) analysis.

<img src="figures/umap_embedding.png" width="500">

The above analysis indicates that (even non-linear) point-wise features are not enough to correctly classify the label, and thus we need to add spatial information/features or and/or learn them directly from the dataset.

##### 3.3 Signal redundancy analysis via FFT pooling

Let's explore a bit more the spatial information content of the images via FFT-based analysis on different scales. The idea is the remove small-scale information repeatedly via FFT-pooling (via Hartley Transform [arxiv.org/abs/1810.04028](https://www.arxiv.org/abs/1810.04028)) and to measure the deterioration between the original image and the "cleaned" image via SSIM loss.

![Figure 9](figures/fft_pooling_effect.png)

__NOTE__: The analysis above demonstrates that a crucial amount of semantic information is contained on small $k$-scales. This information is crucial for detecting clear outlines of rooftops. As the dataset is small, we can include edge filter maps of the RGB image as addtiional model inputs. A quick analysis (shown below) suggests that this additional information is relatively robust to different pooling operations.

<img src="figures/edge_pooling_comparison.png" width="800">

#### 4. Spatial feature engineering

##### 4.1 Edge detection as an additional constraint
Border pixels carry additional information about the semantic structure of input images and are easily computed via morphological operations. Similarly to the inclusion of border information in the prediction task, we can also use the border information of label masks as additional constraints during training.
Following [arxiv.org/pdf/2505.09829](https://www.arxiv.org/abs/2505.09829) we extract the edge pixels of labels via morphological opening and erosion of the label maps. These _edge maps_ may then be used as additional objectives/losses during model training.

<img src="figures/boundary_mask.png" width="600">

### Solution Ideation
Below are the main steps and ingredients listed for the final model configuration.

__Model and training configuration__
+ k-fold CV: 60-20-20 split for training, validation, testing → 4 sets
+ final model is an ensemble of 4 submodels
+ predictions on unseen samples (holdout set) are done via averaging predicted probabilities of all submodels
+ data augmentations: E(2) group + albumentations

__Additional input features__
+ Edge features of input image (Robert's edge filter)

__NN architecture__
+ U-Net
+ Group norm (due to small batch size)
+ (potential) dropout

__Loss configurations and ablation studies__
+ 3 losses: Dice, Focal, IoU
+ Edge label constraint: Additional loss for edge label pixels (see [arxiv.org/pdf/2505.09829](https://www.arxiv.org/abs/2505.09829) for details)

### Results

Below are the final results in tabular form. We compare a total of 6 models: models are either trained wit the Dice, Focal or IoU loss. The `full` models also include an additional loss term comparing the predicted and true label edge maps as described above.

For Focal and IoU loss runs, the full model (with additional edge segmentation loss) consistently outperforms its companion run w/o edge information.
Interestingly, the IoU run also does a better job at minimizing the Dice loss, even though it was trained on the IoU loss.

<img src="figures/metrics_mean_matrix.png" width="600">

We select the IoU-full run as our fiducial model and show the predictions on the holdout set (without control labels) below:

<img src="figures/voting_iou_full_535.png" width="600">
<img src="figures/voting_iou_full_537.png" width="600">
<img src="figures/voting_iou_full_539.png" width="600">
<img src="figures/voting_iou_full_551.png" width="600">
<img src="figures/voting_iou_full_553.png" width="600">

### Future improvements

+ E(2) equivariant network: self-regularized and no need for spatial data aug anymore
+ If domain knowledge available, post-processing step that filters out too small structures, e.g. sheds vs roofs.